# M5 · Validate boundaries and compose a workflow

Contracts check external structure; the domain checks application facts; a workflow retains outcomes. The supplied in-process caller keeps the first exploration free of live-server setup.

**Start:** Before class, from `current-course/teaching-snippets/`, run `uv sync --extra dev` and select its Python environment as the notebook kernel. Notebook-editor/kernel setup is separate classroom preparation. Restart the kernel and run top to bottom. All teaching Python is in the editable cells below; no repository imports or external lesson loaders are needed.

**Explore:** Predict → run → observe → change one input. Expected results assume the supplied values and a fresh run; changing earlier state may change later results. To return to the baseline, restart and rerun from the start.

**Caller used here:** FastAPI’s TestClient uses HTTPX in process. It exercises decoding, routing, and response encoding, but does not establish live-network transport or timeout behavior. External requests are never sent automatically. For the separate-process requests demonstration, use the optional activity linked at the end.

## Exploration 1 — build and observe the mechanism

### 1. State acceptable item values

A boundary contract turns unknown data into a checked interface. It does not silently repair an invalid quantity.

**Before running:** Predict the result or state change.

In [ ]:
from pydantic import BaseModel, ConfigDict, Field, ValidationError

class RequestedItem(BaseModel):
    model_config = ConfigDict(extra="forbid")
    product_id: str = Field(strict=True, min_length=1)
    quantity: int = Field(strict=True, gt=0)

item = RequestedItem(product_id="1", quantity=2)
print(item.model_dump())


**Change and explain:** Which external text conversion must happen before strict integer validation?

<details>
<summary>Reasoned expectation — reveal after predicting</summary>

A valid model supplies named fields with the accepted values.

</details>

### 2. Constrain the whole request

One valid item is not enough when an operation requires a nonempty collection.

**Before running:** Predict the result or state change.

In [ ]:
class OrderRequest(BaseModel):
    model_config = ConfigDict(extra="forbid")
    items: list[RequestedItem] = Field(min_length=1)

request = OrderRequest.model_validate({"items": [{"product_id": "1", "quantity": 2}]})
print(request.items[0].quantity)
print(request.model_dump())


**Change and explain:** Try an empty items list using the handling pattern in the next cell.

<details>
<summary>Reasoned expectation — reveal after predicting</summary>

The accepted request retains the familiar outer items shape.

</details>

### 3. Read rejection at the right field

Validation reports the constraint an input failed. Numeric-looking text and Boolean values are not strict integer quantities here.

**Before running:** Predict the result or state change.

In [ ]:
for quantity in [0, True, "2", 2]:
    try:
        item = RequestedItem(product_id="1", quantity=quantity)
        print("Accepted:", item.quantity)
    except ValidationError as error:
        failure = error.errors()[0]
        print("Rejected:", repr(quantity), failure["loc"], failure["type"])


**Change and explain:** Add -1; explain why ordinary type hints alone would not produce these checks.

<details>
<summary>Reasoned expectation — reveal after predicting</summary>

Only integer 2 is accepted in this list.

</details>

### 4. Describe the response too

A successful status and valid JSON do not establish the response shape. The caller needs its own contract.

**Before running:** Predict the result or state change.

In [ ]:
class OrderLineResponse(BaseModel):
    product_id: str
    name: str
    unit_price_paise: int = Field(strict=True, ge=0)
    quantity: int = Field(strict=True, gt=0)

class OrderResponse(BaseModel):
    order_id: str
    lines: list[OrderLineResponse] = Field(min_length=1)
    total_paise: int = Field(strict=True, ge=0)

print(list(OrderResponse.model_fields))


**Change and explain:** Explain why checking shape still does not independently establish correct total arithmetic.

<details>
<summary>Reasoned expectation — reveal after predicting</summary>

The response requires an identity, nonempty snapshot lines, and a nonnegative integer total.

</details>

### 5. Validate serialized input and a malformed response

Syntax, structure, and business facts are different decisions. Choose a representation for each boundary.

**Before running:** Predict the result or state change.

In [ ]:
message = '{"items":[{"product_id":"1","quantity":2}]}'
validated = OrderRequest.model_validate_json(message)
print(validated.model_dump())
try:
    OrderResponse.model_validate({"order_id": "example", "total_paise": 99800})
except ValidationError as error:
    print("Missing response field:", error.errors()[0]["loc"])
print("Schema properties:", list(OrderRequest.model_json_schema()["properties"]))


**Change and explain:** Explain what the schema cannot establish about product 999.

<details>
<summary>Reasoned expectation — reveal after predicting</summary>

The request validates; the response without lines fails; the schema describes items.

</details>

### 6. Check application facts in the domain

The catalog decides which IDs currently exist. Domain construction calculates and retains purchase facts.

**Before running:** Predict the result or state change.

In [ ]:
from uuid import uuid4
products = {"1": ("USB-C Cable", 49900), "3": ("Water Bottle", 89900)}

def create_order(request):
    lines = []
    for item in request.items:
        name, price = products[item.product_id]
        lines.append({"product_id": item.product_id, "name": name,
                      "unit_price_paise": price, "quantity": item.quantity})
    total = sum(line["unit_price_paise"] * line["quantity"] for line in lines)
    return {"order_id": uuid4().hex, "lines": lines, "total_paise": total}

print(create_order(validated)["total_paise"])


**Change and explain:** Construct a request for 999; explain which later operation rejects it.

<details>
<summary>Reasoned expectation — reveal after predicting</summary>

The checked request yields an order total of 99800 using the catalog’s price.

</details>

## Exploration 2 — change a condition and inspect the consequence

### 7. Connect the checked boundary to the operation

The route receives validated input, delegates current facts, and declares an output contract.

**Before running:** Predict the result or state change.

In [ ]:
from fastapi import FastAPI, HTTPException
from fastapi.testclient import TestClient
app = FastAPI()

@app.post("/orders", response_model=OrderResponse, status_code=201)
def place_order(request: OrderRequest):
    try:
        return create_order(request)
    except KeyError:
        raise HTTPException(404, "Unknown product")

client = TestClient(app)
response = client.post("/orders", json=validated.model_dump())
print(response.status_code, OrderResponse.model_validate(response.json()).total_paise)


**Change and explain:** Try quantity zero and inspect the status before any domain lookup.

<details>
<summary>Reasoned expectation — reveal after predicting</summary>

A valid two-cable request returns 201 and a validated total 99800.

</details>

### 8. Read CSV as text records

A CSV gives named columns, not trusted Python types. An in-memory file keeps this first exploration independent of disk setup.

**Before running:** Predict the result or state change.

In [ ]:
import csv
from io import StringIO
csv_text = "product_id,quantity\n1,2\n3,1\n1,0\n999,1\n1,two\n"
with StringIO(csv_text, newline="") as source:
    rows = list(csv.DictReader(source))
print(rows)
print(type(rows[0]["quantity"]).__name__)


**Change and explain:** Identify which rows have bad source values and which require a catalog check.

<details>
<summary>Reasoned expectation — reveal after predicting</summary>

All quantity values arrive as strings; there are five source rows.

</details>

### 9. Make source conversion explicit

Convert a known external representation before strict validation. Successful conversion does not imply a positive value.

**Before running:** Predict the result or state change.

In [ ]:
def request_from_row(row):
    return OrderRequest(items=[{
        "product_id": row["product_id"],
        "quantity": int(row["quantity"]),
    }])

print(request_from_row(rows[0]).model_dump())
print(request_from_row(rows[3]).model_dump())


**Change and explain:** Explain why product 999 passes this source step.

<details>
<summary>Reasoned expectation — reveal after predicting</summary>

Both shapes validate; only the server’s current catalog can decide membership.

</details>

### 10. Retain the outcome at its boundary

A workflow coordinates steps. Source failure, HTTP rejection, and accepted response are distinct outcomes.

**Before running:** Predict the result or state change.

In [ ]:
def process_row(row):
    try:
        request = request_from_row(row)
    except ValueError as error:
        return {"status": "invalid_source", "detail": str(error)}
    response = client.post("/orders", json=request.model_dump())
    if response.status_code >= 400:
        return {"status": "rejected", "http_status": response.status_code}
    order = OrderResponse.model_validate(response.json())
    return {"status": "created", "order_id": order.order_id}

print(process_row(rows[0])["status"])
print(process_row(rows[2])["status"])
print(process_row(rows[3])["status"])


**Change and explain:** This in-process path does not simulate transport failure; M8 supplies controlled external failures.

<details>
<summary>Reasoned expectation — reveal after predicting</summary>

created, invalid_source, rejected. Accepted rows each create their own order.

</details>

### 11. Process every row and keep its source position

The batch retains evidence instead of losing later rows after an expected source rejection.

**Before running:** Predict the result or state change.

In [ ]:
from collections import Counter
import logging
logger = logging.getLogger("pico_zon.workflow")
outcomes = []
for row_number, row in enumerate(rows, start=2):
    outcome = {"row": row_number, **process_row(row)}
    outcomes.append(outcome)
    logger.info("Row %s: %s", row_number, outcome["status"])
    print("Row", row_number, outcome["status"])
print(dict(Counter(item["status"] for item in outcomes)))


**Change and explain:** Why do progress messages and a retained report have different lifetimes?

<details>
<summary>Reasoned expectation — reveal after predicting</summary>

Two created, two invalid-source, one rejected. Logger output depends on the editor’s configured handlers.

</details>

### 12. Write and inspect the retained report

A report outlives the workflow. Use a throwaway folder here; the lab writes its intended report file.

**Before running:** Predict the result or state change.

In [ ]:
import json
from pathlib import Path
from tempfile import TemporaryDirectory
with TemporaryDirectory() as directory:
    report_path = Path(directory) / "order-report.json"
    report_path.write_text(json.dumps(outcomes, indent=2), encoding="utf-8")
    retained = json.loads(report_path.read_text(encoding="utf-8"))
    print(len(retained), retained[0]["status"])
client.close()


**Change and explain:** Derive counts from the report instead of hard-coding the sample expectations.

<details>
<summary>Reasoned expectation — reveal after predicting</summary>

Five outcomes were retained and readable. The temporary folder is removed after inspection.

</details>

## Apply the mechanism in Nano-Zon

Follow the [M5 lab guide](../../projects/nano-zon/labs/m05/README.md). The codealong isolates the mechanism; the lab combines it with the supplied application.

State the principle, point to the code that expresses it, then explain one limit of the observed result.

### Optional live-server activity

The [teaching-snippet guide](../../teaching-snippets/README.md#api-sequence) supplies the matching uvicorn server and separate Python caller. This activity requires two terminals and deliberate launch; it is not a prerequisite for the cells above. External order submissions can create additional orders. A timeout does not establish whether the server created an order, so do not blindly repeat a POST.

**Authoring status:** This notebook contains editable teaching source, not captured execution output. Syntax and structure can be inspected independently of execution; runtime delivery review remains pending.